### This is an experiment for new image-text retrieval web ###
NOTES:
- This experiment uses Apple's version of CLIP, namely "DFN5B CLIP ViT H14"
- The following part is to create a vector collection on Zilliz Vector Database, with the help of Milvus

**SETUP**

In [1]:
import torch
from tqdm import tqdm
from pprint import pprint
from pymilvus import MilvusClient, DataType, CollectionSchema, FieldSchema

**DATASET LOAD**

In [16]:
full_dataset = torch.load('dataset-aic2025-with-feature-b1.pt')
data_insert = list(full_dataset.values())  # Data to be inserted into Zilliz Cloud is values of dataset dictionary

In [ ]:
pprint(data_insert[123456])

**CONNECT TO CLIENT**

In [3]:
client = MilvusClient(
    uri='https://in03-cbdb10d1d199984.serverless.aws-eu-central-1.cloud.zilliz.com',
    token='6305ff67695e59bf211ca717cb68f74f7367ccfd56195824ba6aad7c07f5924cc6c9da8aadec4354aedc193a997225494903a9d7'
)

**CREATE COLLECTION'S SCHEMA**

In [18]:
#Schema build
id = FieldSchema(
    'id',
    dtype=DataType.INT64,
    is_primary=True,
    auto_id=True
)
vector = FieldSchema(
    'vector',
    dtype=DataType.FLOAT_VECTOR,
    dim=1024,
    description='Feature vectors obtained from DFN5B-CLIP-ViT-H14 computation.'
)
img_key = FieldSchema(
    'img_key',
    dtype=DataType.VARCHAR,
    max_length=1024,
    description='Image key is obtained from AWS S3 bucket and used to display images on the main web.'
)

video_id = FieldSchema(
    'video_id',
    dtype=DataType.VARCHAR,
    max_length=32,
    is_partition_key=True,
    description='The ID of the orignal video, where the frame belongs to.'
)

frame_id = FieldSchema(
    'frame_id',
    dtype=DataType.VARCHAR,
    max_length=16,
    description='The ID as the name of the image given by contest organizers.'
)

time_order = FieldSchema(
    'time_order',
    dtype=DataType.VARCHAR,
    max_length=16,
    description='The time where the image lies in the original video, in [ms].'
)

frame_order = FieldSchema(
    'frame_order',
    dtype=DataType.VARCHAR,
    max_length=16,
    description='The n-th frame of the video.'
)

answer_key = FieldSchema(
    'answer_key',
    dtype=DataType.VARCHAR,
    max_length=256,
    description='The answer key to be sent via API calls. Only used in Finals.'
)

youtube_link = FieldSchema(
    'youtube_link',
    dtype=DataType.VARCHAR,
    max_length=1024,
    description='The YouTube link of the video where this image belong to.'
)

publish_date = FieldSchema(
    'publish_date',
    dtype=DataType.VARCHAR,
    max_length=32,
    description='The publish date of the original video. This is mostly untouched.'
)

schema = CollectionSchema(
    fields=[
        id,
        vector,
        img_key,
        video_id,
        frame_id,
        time_order,
        frame_order,
        answer_key,
        youtube_link,
        publish_date
    ],
    description='This schema gives most of the essential details for querying and filtering images. Made specifically for AIC.'
)

**PREPARE NEW INDEX**

In [17]:
index_params = MilvusClient.prepare_index_params()

index_params.add_index(
    field_name='vector',
    metric_type='COSINE',
    index_type='HNSW',
    index_name='vector_index',
    params={
        'M': 32,
        'efConstruction': 200
    }
)

**CREATE NEW COLLECTION**

In [23]:
if client.has_collection(collection_name="aic2025"):
    client.drop_collection(collection_name="aic2025")
client.create_collection(
    collection_name="aic2025",
    schema=schema,
    index_params=index_params,
    enable_dynamic_field=True,
    consistency_level='Eventually'
)

**BATCHING**

In [20]:
BATCH_SIZE = 12288
batches = []
for i in range(0, len(data_insert), BATCH_SIZE):
    batches.append(data_insert[i:i + BATCH_SIZE] if i < len(data_insert) else data_insert[i:len(data_insert)])

print(f'Done batching. Batch count: {len(batches)}')

Done batching. Batch count: 15


**INSERT VECTORS TO COLLECTION**

In [24]:
with tqdm(batches, desc='Inserting batches to vector database') as t:
    for batch in t:
        client.insert(
            'aic2025',
            data=batch
        )

        elapsed = t.format_dict['elapsed']
        elapsed_str = t.format_interval(elapsed)

Inserting batches to vector database: 100%|██████████| 15/15 [10:46<00:00, 43.11s/it]


**DEBUGGING**

In [14]:
image_features = np.squeeze(image_features)
print(image_features)

[[ 0.01472929  0.05896474  0.00679139 ... -0.01624357  0.03813367
  -0.04635565]
 [-0.04977068  0.05377883  0.00242164 ... -0.03778296  0.01258671
   0.00754181]
 [-0.03032801  0.06692141  0.00739449 ... -0.04433     0.03128408
   0.01230146]
 ...
 [-0.02354688  0.06325173  0.00765806 ... -0.03973793  0.02035062
   0.00969038]
 [-0.03171886  0.06158489  0.00196003 ... -0.037081    0.01561518
   0.00747865]
 [-0.03978864  0.04160773  0.00909726 ... -0.04254298  0.00655768
  -0.0027554 ]]
